# 04b: LOCK v2 (validation only, three seeds, fixed referral burden)

Replaces the superseded cost-optimal lock. Loads **only** `eyepacs_predictions_val_*`
files (33 = 11 architectures x 3 seeds). No test or APTOS file is opened.

## What is locked

**Cost function.** Miss weights for referable grades only: grade 4 = 500, grade 3 = 200,
grade 2 = 15; false positive = 1. (v1's grade-1 weight never fired; removed.)

**Operating point.** Fixed referral burden: **25% primary** (IDx-DR pivotal operating
characteristics at EyePACS prevalence); curve from 10% to 40% in 2.5-point steps. At
burden r on n images, every configuration refers exactly round(r x n) cases, chosen as
its highest-scoring cases. Label-free.

**OR-gate (primary, rank-based).** Each model's probabilities are converted to ranks
within the evaluation set; the OR-gate score is max(rank_A, rank_B). Equivalent to running
both devices at the same individual referral rate and referring if either flags, so a
more liberally calibrated model cannot dominate (Reviewer 3). Sensitivity analysis:
max of raw probabilities.

**Comparators at each burden.** (a) the better of the pair's own two constituents;
(b) the best single architecture in the whole roster (lowest mean cost over its three
seeds). Both chosen on the evaluation set, i.e. conservatively in favour of the single
model. Only (b) answers the deployment question.

**Seeds.** Every pair of architectures has 9 seed combinations (3 x 3); results average
over them. The engineering pair uses **matched** replicates only (3 combinations), because
each adversarial model was trained against its own replicate's DenseNet.

**Tiers.** (1) Operation, primary: number of the 55 pairs whose seed-averaged advantage over
comparator (a) is positive at 25%, plus the mean advantage across all 55. (2) Witness: the
pair with the largest seed-averaged advantage over (a) at 25% on validation. (3)
Engineering: densenet121_binary + retfound_adversarial, pre-specified.

**Inference (applied at evaluation).** Patient-clustered bootstrap on EyePACS (patients
resampled jointly across all models), image-level on APTOS. Tier 1 claim is supported if the
95% interval of the mean advantage across the 55 pairs excludes zero.

CPU runtime. Run cells 1-6; nothing is written until Cell 6.

In [1]:
# Cell 1: auth; load the 33 VALIDATION prediction files only
import io, os, json, hashlib, datetime, itertools
import numpy as np, pandas as pd
from scipy.stats import rankdata
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_RESULTS     = '1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'

ARCHS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
         'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
         'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = [0, 1, 2]
def tag(k): return '' if k == 0 else f'_seed{k}'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done = False
        while not done: _, done = d.next_chunk()
def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def up(path, folder, mime):
    svc.files().create(body={'name': os.path.basename(path), 'parents': [folder]},
                       media_body=MediaFileUpload(path, mimetype=mime), fields='id').execute()
    print('  uploaded', os.path.basename(path))
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

if find_id('locked_config.json', F_RESULTS):
    raise SystemExit('locked_config.json already exists. The lock is immutable; do not re-lock.')

P, HASHES, ref = {}, {}, None
for a in ARCHS:
    for k in SEEDS:
        fn = f'eyepacs_predictions_val_{a}{tag(k)}_v2.csv'
        assert fn.startswith('eyepacs_predictions_val_'), 'safeguard: validation files only'
        fid = find_id(fn, F_PREDICTIONS)
        if not fid: raise FileNotFoundError(fn)
        dl(fid, fn); d = pd.read_csv(fn); HASHES[fn] = sha256(fn)
        assert set(d['split']) == {'val'}, f'{fn}: non-validation rows'
        if ref is None: ref = d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
        assert (d['image_id'].astype(str).values == ref['image_id'].astype(str).values).all(), f'{fn}: row order'
        P[(a, k)] = d['predicted_probability'].to_numpy(np.float64)

LAB = ref['true_label'].to_numpy(int); GRD = ref['true_grade'].to_numpy(int); N = len(LAB)
assert ((LAB == 1) == (GRD >= 2)).all()
print(f'loaded {len(P)} validation files | {N} images, {ref.patient_id.nunique()} patients, '
      f'referable {LAB.sum()} ({LAB.mean():.3f}), severe {(GRD >= 3).sum()}')

loaded 33 validation files | 2900 images, 1450 patients, referable 565 (0.195), severe 132


In [2]:
# Cell 2: QC (validation AUC recomputed from the files)
from sklearn.metrics import roc_auc_score
qc = pd.DataFrame([{'arch': a, 'seed': k, 'val_auc': roc_auc_score(LAB, P[(a, k)])} for a in ARCHS for k in SEEDS])
tbl = qc.pivot(index='arch', columns='seed', values='val_auc').round(4)
tbl['mean'] = tbl.mean(axis=1).round(4); tbl['range'] = (tbl[[0,1,2]].max(axis=1) - tbl[[0,1,2]].min(axis=1)).round(4)
print(tbl.sort_values('mean', ascending=False).to_string())

seed                         0       1       2    mean   range
arch                                                          
retfound_binary         0.8798  0.8788  0.8814  0.8800  0.0026
densenet121_5class      0.8810  0.8745  0.8826  0.8794  0.0081
retfound_adversarial    0.8771  0.8793  0.8813  0.8792  0.0042
densenet121_binary      0.8776  0.8801  0.8796  0.8791  0.0025
retfound_5class         0.8764  0.8778  0.8789  0.8777  0.0025
vit_base_binary         0.8683  0.8710  0.8653  0.8682  0.0057
vit_base_5class         0.8632  0.8714  0.8663  0.8670  0.0082
efficientnet_b3_5class  0.8553  0.8610  0.8557  0.8573  0.0057
resnet50_binary         0.8426  0.8583  0.8613  0.8541  0.0187
efficientnet_b3_binary  0.8534  0.8527  0.8554  0.8538  0.0027
resnet50_5class         0.8532  0.8559  0.8464  0.8518  0.0095


In [3]:
# Cell 3: locked definitions
W = {'w4': 500, 'w3': 200, 'w2': 15, 'wfp': 1}
MISSW = np.where(LAB == 1, np.where(GRD == 4, W['w4'], np.where(GRD == 3, W['w3'], W['w2'])), 0).astype(float)
NEG = (LAB == 0).astype(float); SEV = ((LAB == 1) & (GRD >= 3)).astype(float)
assert not ((LAB == 1) & (GRD == 1)).any(), 'grade-1 eye labelled referable'

PRIMARY_BURDEN = 0.25
BURDENS = [round(x, 3) for x in np.arange(0.10, 0.4001, 0.025)]
RS = {r: int(round(r * N)) for r in BURDENS}

def curve(score):
    """Cost and severe misses when referring the top-R cases by score, for every burden. Label-free selection."""
    order = np.argsort(-score, kind='stable')
    cm, cf, cs = np.cumsum(MISSW[order]), np.cumsum(NEG[order]), np.cumsum(SEV[order])
    idx = np.array([RS[r] - 1 for r in BURDENS])
    return (MISSW.sum() - cm[idx]) + W['wfp'] * cf[idx], SEV.sum() - cs[idx]

RANK = {key: rankdata(p) / N for key, p in P.items()}
def or_score(a, i, b, j, mode='rank'):
    return np.maximum(RANK[(a, i)], RANK[(b, j)]) if mode == 'rank' else np.maximum(P[(a, i)], P[(b, j)])
print('burdens:', BURDENS, '| referral counts:', [RS[r] for r in BURDENS])

burdens: [np.float64(0.1), np.float64(0.125), np.float64(0.15), np.float64(0.175), np.float64(0.2), np.float64(0.225), np.float64(0.25), np.float64(0.275), np.float64(0.3), np.float64(0.325), np.float64(0.35), np.float64(0.375), np.float64(0.4)] | referral counts: [290, 362, 435, 507, 580, 652, 725, 798, 870, 942, 1015, 1088, 1160]


In [4]:
# Cell 4: validation results at every burden, all seed combinations
SINGLE = {key: curve(p) for key, p in P.items()}               # (cost array, severe-miss array)
ARCH_MEAN = {a: np.mean([SINGLE[(a, k)][0] for k in SEEDS], axis=0) for a in ARCHS}
BEST_ROSTER = np.min(np.vstack([ARCH_MEAN[a] for a in ARCHS]), axis=0)
BEST_ROSTER_NAME = [ARCHS[i] for i in np.argmin(np.vstack([ARCH_MEAN[a] for a in ARCHS]), axis=0)]

rows = []
for a, b in itertools.combinations(ARCHS, 2):
    for i in SEEDS:
        for j in SEEDS:
            for mode in ('rank', 'raw'):
                c_or, s_or = curve(or_score(a, i, b, j, mode))
                ca, cb = SINGLE[(a, i)][0], SINGLE[(b, j)][0]
                for t, r in enumerate(BURDENS):
                    better = min(ca[t], cb[t])
                    rows.append({'model_a': a, 'seed_a': i, 'model_b': b, 'seed_b': j, 'mode': mode, 'burden': r,
                                 'or_cost': c_or[t], 'or_severe_misses': int(s_or[t]),
                                 'better_constituent_cost': better, 'best_roster_cost': BEST_ROSTER[t],
                                 'adv_vs_constituent': better - c_or[t],
                                 'adv_vs_roster': BEST_ROSTER[t] - c_or[t]})
LONG = pd.DataFrame(rows)
PAIR = (LONG.groupby(['model_a','model_b','mode','burden'])
            .agg(or_cost=('or_cost','mean'), adv_vs_constituent=('adv_vs_constituent','mean'),
                 adv_vs_roster=('adv_vs_roster','mean'),
                 combos_winning=('adv_vs_constituent', lambda x: int((x > 0).sum())))
            .reset_index())
print(f'{len(LONG)} rows (55 pairs x 9 seed combos x 2 modes x {len(BURDENS)} burdens)')

12870 rows (55 pairs x 9 seed combos x 2 modes x 13 burdens)


In [5]:
# Cell 5: tiers on validation; engineering pair; rho
P25 = PAIR[(PAIR['mode'] == 'rank') & (PAIR['burden'] == PRIMARY_BURDEN)].sort_values('adv_vs_constituent', ascending=False)
t_primary = BURDENS.index(PRIMARY_BURDEN)

tier1_val = {'pairs_winning_of_55': int((P25['adv_vs_constituent'] > 0).sum()),
             'mean_adv_vs_constituent': round(float(P25['adv_vs_constituent'].mean()), 2),
             'seed_combos_winning_of_495': int(P25['combos_winning'].sum()),
             'pairs_beating_best_roster_single_of_55': int((P25['adv_vs_roster'] > 0).sum()),
             'best_roster_single_at_25': BEST_ROSTER_NAME[t_primary]}

W_ = P25.iloc[0]
witness = {'model_a': W_.model_a, 'model_b': W_.model_b,
           'validation_adv_vs_constituent': round(float(W_.adv_vs_constituent), 2),
           'validation_adv_vs_roster': round(float(W_.adv_vs_roster), 2),
           'validation_seed_combos_winning_of_9': int(W_.combos_winning)}

ENG = ('densenet121_binary', 'retfound_adversarial')
eng = LONG[(LONG.model_a == ENG[0]) & (LONG.model_b == ENG[1]) & (LONG['mode'] == 'rank') &
           (LONG.burden == PRIMARY_BURDEN) & (LONG.seed_a == LONG.seed_b)]
engineering = {'model_a': ENG[0], 'model_b': ENG[1], 'seed_pairing': 'matched replicates (k with k)',
               'validation_adv_vs_constituent_per_seed': [round(float(x), 2) for x in eng['adv_vs_constituent']],
               'validation_mean_adv_vs_constituent': round(float(eng['adv_vs_constituent'].mean()), 2)}

# rho: phi correlation of error indicators, each model at its own 25% referral, averaged over seed combos
R25 = RS[PRIMARY_BURDEN]
def errs(p):
    pred = np.zeros(N, int); pred[np.argsort(-p, kind='stable')[:R25]] = 1
    return (pred != LAB).astype(float)
ERR = {key: errs(p) for key, p in P.items()}
rho = {f'{a}|{b}': round(float(np.mean([np.corrcoef(ERR[(a, i)], ERR[(b, j)])[0, 1] for i in SEEDS for j in SEEDS])), 4)
       for a, b in itertools.combinations(ARCHS, 2)}

curve_val = (PAIR[PAIR['mode'] == 'rank'].groupby('burden')
             .agg(pairs_winning=('adv_vs_constituent', lambda x: int((x > 0).sum())),
                  mean_adv=('adv_vs_constituent', 'mean'),
                  pairs_beating_roster=('adv_vs_roster', lambda x: int((x > 0).sum()))).round(2))

print('TIER 1 (validation, 25%, rank OR):', tier1_val)
print('TIER 2 witness:', witness)
print('TIER 3 engineering:', engineering)
print('\ncurve on validation:'); print(curve_val.to_string())
print('\ntop 5 pairs at 25%:')
print(P25.head(5)[['model_a','model_b','or_cost','adv_vs_constituent','adv_vs_roster','combos_winning']].round(1).to_string(index=False))

TIER 1 (validation, 25%, rank OR): {'pairs_winning_of_55': 50, 'mean_adv_vs_constituent': 498.04, 'seed_combos_winning_of_495': 400, 'pairs_beating_best_roster_single_of_55': 34, 'best_roster_single_at_25': 'densenet121_binary'}
TIER 2 witness: {'model_a': 'vit_base_5class', 'model_b': 'retfound_adversarial', 'validation_adv_vs_constituent': 1235.44, 'validation_adv_vs_roster': 708.67, 'validation_seed_combos_winning_of_9': 9}
TIER 3 engineering: {'model_a': 'densenet121_binary', 'model_b': 'retfound_adversarial', 'seed_pairing': 'matched replicates (k with k)', 'validation_adv_vs_constituent_per_seed': [565.0, 428.0, 741.0], 'validation_mean_adv_vs_constituent': 578.0}

curve on validation:
        pairs_winning  mean_adv  pairs_beating_roster
burden                                               
0.100              40    564.61                    24
0.125              36    151.06                     2
0.150              42    299.46                    14
0.175              47    432.

In [6]:
# Cell 6: write the immutable lock
LOCK = {
  'locked_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'supersedes': 'SUPERSEDED_locked_config_cost-optimal-thresholds.json (cost-optimal operating points referred 46-95%)',
  'locked_on': 'EyePACS validation predictions only, 33 files; no test or APTOS file opened',
  'validation_file_sha256': HASHES,
  'cost_function': {'weights': W, 'grade1_note': 'v1 grade-1 weight never fired (grade 1 is label 0); removed'},
  'operating_point': {'primary_burden': PRIMARY_BURDEN, 'curve_burdens': BURDENS,
                      'anchor': 'IDx-DR pivotal trial (Abramoff 2018): sens 87.2%, spec 90.7% at EyePACS prevalence ~ 24.5% referral',
                      'selection': 'refer top round(r*n) cases by score; label-free'},
  'or_gate': {'primary': 'rank-based: max of within-set ranks (both devices at the same individual referral rate)',
              'sensitivity': 'raw-probability max'},
  'comparators': {'a_constituent': 'better of the two constituents at the same burden (chosen on evaluation set)',
                  'b_roster': 'best single architecture by mean cost over 3 seeds at the same burden (chosen on evaluation set)'},
  'seeds': {'replicates': SEEDS, 'pairs': 'all 9 seed combinations averaged',
            'engineering': 'matched replicates only (3)'},
  'tier1_operation': {'primary': True, 'statistic': 'pairs of 55 with positive seed-averaged advantage vs comparator (a) at 25%; '
                      'and mean advantage across all 55', 'claim_supported_if': '95% bootstrap CI of mean advantage excludes zero',
                      'evaluate_on': ['eyepacs_test (25% and curve)', 'aptos_external (curve only)'],
                      'validation': tier1_val},
  'tier2_witness': witness,
  'tier3_engineering': engineering,
  'rho': {'definition': 'phi correlation of error indicators, each model at its own 25% referral, averaged over 9 seed combos; '
                        'measured on validation, concordant severe misses counted on test', 'validation': rho},
  'weight_invariance_grid': {'miss_weight_sets': [[500,200,15],[250,100,10],[1000,400,20],[100,50,10],[50,25,5]],
                             'false_positive_weights': [1, 5, 20, 50, 100]},
  'inference': {'eyepacs': 'patient-clustered bootstrap, 2000 resamples, patients resampled jointly across all models',
                'aptos': 'image-level bootstrap (no patient ids)', 'training_variance': 'three seeds per architecture'},
}
with open('locked_config.json', 'w') as f: json.dump(LOCK, f, indent=2, default=str)
PAIR.to_csv('eyepacs_pairs_validation_allburdens_v2.csv', index=False)
up('locked_config.json', F_RESULTS, 'application/json')
up('eyepacs_pairs_validation_allburdens_v2.csv', F_RESULTS, 'text/csv')
print('LOCKED at', LOCK['locked_at_utc'], '\nDo not re-run. Evaluation (05) reads this lock.')

  uploaded locked_config.json
  uploaded eyepacs_pairs_validation_allburdens_v2.csv
LOCKED at 2026-09-22T15:40:58.674050+00:00 
Do not re-run. Evaluation (05) reads this lock.
